# TransFASNet 

Single-branch Transformer encoder with contrastive pretraining for credit card fraud detection.

Protocol: random stratified 80/10/10 split, BorderlineSMOTE on the training partition, pretraining windows built from the full scaled dataset, 10 pretraining epochs, 10 fine-tuning epochs.

Dataset: place `creditcard.csv` at the path set in `FILE_PATH` below, or update it to a local path. Results from a completed run are stored separately in `results/original-run/`.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import multiprocessing as mp
# Use spawn to avoid multiprocessing issues with CUDA/fork in notebooks
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import BorderlineSMOTE

## Hyperparameters and device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE_PRETRAIN = 512
BATCH_SIZE_FINETUNE = 256
SEQ_LEN = 8             # sliding window length
EMBED_DIM = 128
TRANSFORMER_LAYERS = 3
NUM_HEADS = 4
MLP_DIM = 256
PROJ_DIM = 64           # contrastive projection head dim
PRETRAIN_EPOCHS = 10
FINETUNE_EPOCHS = 10
LR_PRETRAIN = 3e-4
LR_FINETUNE = 1e-4
TEMP = 0.1              # contrastive temperature
ALPHA_CONTRAST = 1.0
ALPHA_TEMPORAL = 1.0
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Load and preprocess data

Note: `FILE_PATH` below points to a Google Drive location for Colab. For a local run, change it to the path of `creditcard.csv` on your machine.

In [ ]:
FILE_PATH = './data/creditcard.csv'
df = pd.read_csv(FILE_PATH)

# Keep numeric columns only 
df_numeric = df.select_dtypes(include=['number']).copy()

# Ensure 'Time' exists; otherwise create an index-based time
if 'Time' not in df_numeric.columns:
    df_numeric['Time'] = np.arange(len(df_numeric))

# Sort by Time to build windows (temporal structure)
df_numeric = df_numeric.sort_values('Time').reset_index(drop=True)

y = df_numeric['Class'].values
X = df_numeric.drop('Class', axis=1).values
feature_names = df_numeric.drop('Class', axis=1).columns.tolist()

# scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/valid/test split (80/10/10 split)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, shuffle=True, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, shuffle=True, stratify=y_temp
)

# For fine-tuning we'll oversample the training set like your original pipeline
sm = BorderlineSMOTE(kind='borderline-1', random_state=SEED)
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)

## Sliding windows for pretraining

In [ ]:
def sliding_windows(arr, seq_len):
    N = arr.shape[0]
    if N < seq_len + 1:
        return np.zeros((0, seq_len, arr.shape[1]))
    out = np.array([arr[i:i+seq_len] for i in range(0, N - seq_len + 1)])
    return out

ALL_WINDOWS = sliding_windows(X_scaled, SEQ_LEN)  # shape (num_windows, SEQ_LEN, D)
print("Number of pretraining windows:", ALL_WINDOWS.shape[0])

## Augmentations for contrastive views

In [ ]:
def augment_noise(x, sigma=0.02):
    return x + np.random.normal(0, sigma, size=x.shape)

def augment_masking(x, mask_prob=0.15):
    x2 = x.copy()
    mask = np.random.rand(*x2.shape) < mask_prob
    x2[mask] = 0.0
    return x2

def make_view(x):
    x_aug = augment_noise(x, sigma=0.02)
    x_aug = augment_masking(x_aug, mask_prob=0.12)
    return x_aug

## Dataset classes

In [ ]:
class PretrainDataset(Dataset):
    def __init__(self, windows):
        # ensure float32
        self.windows = windows.astype(np.float32)

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        seq = self.windows[idx]            # (SEQ_LEN, D)
        view1 = make_view(seq)
        view2 = make_view(seq)
        next_target = seq[-1].copy()
        return view1, view2, next_target


class FinetuneDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## DataLoaders

In [ ]:
# Use num_workers=0 in notebooks/Colab to avoid multiprocessing teardown issues.
# If you move to a script on Linux and want faster IO, you can set num_workers>0
# and keep the mp.set_start_method('spawn') above.
num_workers = 0
pin_memory = True if torch.cuda.is_available() else False

pretrain_ds = PretrainDataset(ALL_WINDOWS)
pretrain_loader = DataLoader(pretrain_ds, batch_size=BATCH_SIZE_PRETRAIN,
                             shuffle=True, drop_last=True, num_workers=num_workers,
                             pin_memory=pin_memory)

finetune_train_ds = FinetuneDataset(X_train_over, y_train_over)
finetune_valid_ds = FinetuneDataset(X_valid, y_valid)
finetune_test_ds  = FinetuneDataset(X_test, y_test)

finetune_train_loader = DataLoader(finetune_train_ds, batch_size=BATCH_SIZE_FINETUNE,
                                   shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
finetune_valid_loader = DataLoader(finetune_valid_ds, batch_size=BATCH_SIZE_FINETUNE,
                                   shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
finetune_test_loader  = DataLoader(finetune_test_ds, batch_size=BATCH_SIZE_FINETUNE,
                                   shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

## Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class TransFASNet(nn.Module):
    def __init__(self, input_dim, seq_len, embed_dim=EMBED_DIM, n_layers=TRANSFORMER_LAYERS, n_heads=NUM_HEADS, mlp_dim=MLP_DIM, proj_dim=PROJ_DIM, num_classes=2):
        super().__init__()
        self.input_dim = input_dim
        self.seq_len = seq_len
        self.embed = nn.Linear(input_dim, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=mlp_dim, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.proj_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, proj_dim)
        )
        self.temporal_head = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(mlp_dim, num_classes)
        )

    def forward_backbone(self, x):  # x: (B, seq_len, D)
        h = self.embed(x)            # (B, seq_len, embed)
        h = self.pos_enc(h)
        h = self.transformer(h)      # (B, seq_len, embed)
        z = h.mean(dim=1)            # (B, embed)
        return z

    def project(self, z):
        return self.proj_head(z)

    def temporal_predict(self, z):
        return self.temporal_head(z)

    def classify(self, z):
        return self.classifier(z)

    def forward_classify_from_flat(self, x_flat):
        z = self.embed(x_flat.unsqueeze(1))  # (B,1,embed)
        z = self.pos_enc(z)
        z = self.transformer(z)
        z = z.mean(dim=1)
        logits = self.classify(z)
        return logits

# instantiate model
input_dim = X_train.shape[1]
model = TransFASNet(input_dim=input_dim, seq_len=SEQ_LEN).to(DEVICE)

## Losses

In [ ]:
def nt_xent_loss_simple(z1, z2, temperature=TEMP):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    batch_size = z1.size(0)
    representations = torch.cat([z1, z2], dim=0)  # 2B x D
    similarity = torch.matmul(representations, representations.T)  # 2B x 2B
    mask = (~torch.eye(2*batch_size, 2*batch_size, dtype=torch.bool)).to(DEVICE)
    sim = similarity / temperature
    exp_sim = torch.exp(sim) * mask.float()
    denom = exp_sim.sum(dim=1)
    positives = torch.cat([torch.diag(similarity, batch_size), torch.diag(similarity, -batch_size)], dim=0) / temperature
    loss = - (positives - torch.log(denom))
    return loss.mean()

## Pretraining loop

In [ ]:
optimizer_pre = torch.optim.Adam(model.parameters(), lr=LR_PRETRAIN)

model.train()
for epoch in range(PRETRAIN_EPOCHS):
    pbar = tqdm(pretrain_loader, desc=f"Pretrain Epoch {epoch+1}/{PRETRAIN_EPOCHS}")
    epoch_loss = 0.0
    for view1, view2, next_target in pbar:
        # ensure float32 and move to device
        view1 = torch.tensor(view1, dtype=torch.float32) if not isinstance(view1, torch.Tensor) else view1.float()
        view2 = torch.tensor(view2, dtype=torch.float32) if not isinstance(view2, torch.Tensor) else view2.float()
        next_target = torch.tensor(next_target, dtype=torch.float32) if not isinstance(next_target, torch.Tensor) else next_target.float()

        view1 = view1.to(DEVICE)
        view2 = view2.to(DEVICE)
        next_target = next_target.to(DEVICE)

        # forward
        z1 = model.forward_backbone(view1)
        z2 = model.forward_backbone(view2)

        # projection for contrastive loss
        p1 = model.project(z1)
        p2 = model.project(z2)

        # losses
        c_loss = nt_xent_loss_simple(p1, p2, temperature=TEMP)
        pred_next = model.temporal_predict(z1)
        t_loss = F.mse_loss(pred_next, next_target)

        loss = ALPHA_CONTRAST * c_loss + ALPHA_TEMPORAL * t_loss

        # backward
        optimizer_pre.zero_grad()
        loss.backward()
        optimizer_pre.step()

        epoch_loss += loss.item()
        pbar.set_postfix({'loss': epoch_loss / (pbar.n + 1e-9)})

    print(f"Epoch {epoch+1} pretrain loss: {epoch_loss / len(pretrain_loader):.6f}")

## Fine-tuning loop

In [ ]:
optimizer_ft = torch.optim.Adam(model.parameters(), lr=LR_FINETUNE)
criterion = nn.CrossEntropyLoss()

def evaluate_classifier(loader):
    model.eval()
    preds = []
    probs = []
    trues = []
    with torch.no_grad():
        for Xb, yb in loader:
            # ensure float and move to device
            Xb = torch.tensor(Xb, dtype=torch.float32) if not isinstance(Xb, torch.Tensor) else Xb.float()
            Xb = Xb.to(DEVICE)
            yb = torch.tensor(yb, dtype=torch.int64) if not isinstance(yb, torch.Tensor) else yb.long()
            yb = yb.to(DEVICE)

            logits = model.forward_classify_from_flat(Xb)
            prob = torch.softmax(logits, dim=1)
            out = torch.argmax(prob, dim=1)

            preds.append(out.cpu().numpy())
            probs.append(prob.cpu().numpy())
            trues.append(yb.cpu().numpy())

    preds = np.concatenate(preds)
    probs = np.concatenate(probs)
    trues = np.concatenate(trues)

    acc = (preds == trues).mean()
    from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
    prec = precision_score(trues, preds, zero_division=0)
    rec = recall_score(trues, preds, zero_division=0)
    f1 = f1_score(trues, preds, zero_division=0)
    try:
        roc = roc_auc_score(trues, probs[:, 1]) if probs.shape[1] == 2 else roc_auc_score(trues, probs, multi_class='ovr')
    except:
        roc = 0.0
    return acc, prec, rec, f1, roc

# fine-tune loop
best_val_f1 = 0.0
for epoch in range(FINETUNE_EPOCHS):
    model.train()
    pbar = tqdm(finetune_train_loader, desc=f"Finetune Epoch {epoch+1}/{FINETUNE_EPOCHS}")
    running_loss = 0.0
    for Xb, yb in pbar:
        Xb = torch.tensor(Xb, dtype=torch.float32) if not isinstance(Xb, torch.Tensor) else Xb.float()
        yb = torch.tensor(yb, dtype=torch.int64) if not isinstance(yb, torch.Tensor) else yb.long()

        Xb = Xb.to(DEVICE)
        yb = yb.to(DEVICE)

        logits = model.forward_classify_from_flat(Xb)
        loss = criterion(logits, yb)
        optimizer_ft.zero_grad()
        loss.backward()
        optimizer_ft.step()
        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1e-9)})

    val_acc, val_prec, val_rec, val_f1, val_roc = evaluate_classifier(finetune_valid_loader)
    print(f"Val acc {val_acc:.4f} prec {val_prec:.4f} rec {val_rec:.4f} f1 {val_f1:.4f} roc {val_roc:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_transfas_net.pth")
        print("Saved best model.")

## Test evaluation

In [ ]:
model.load_state_dict(torch.load("best_transfas_net.pth", map_location=DEVICE))
test_acc, test_prec, test_rec, test_f1, test_roc = evaluate_classifier(finetune_test_loader)
print("===== Test results =====")
print(f"Acc: {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall: {test_rec:.4f}")
print(f"F1: {test_f1:.4f}")
print(f"ROC AUC: {test_roc:.4f}")